# Argumentation graduée — sémantiques de classement (h-Categoriser & fardeau)

Dans le notebook compagnon [`Argument_Analysis_Dung_AF_Semantics`](Argumentation-03-Dung-AF-Semantics-Python.ipynb),
les sémantiques de Dung (fondée, préférée, stable) répondent par **oui ou non** : un argument est
*dans* une extension, ou il n'y est pas. C'est une vision **tout-ou-rien** de l'acceptabilité.

Les **sémantiques de classement** (*ranking-based / gradual semantics*) répondent à une question plus fine :
*« à quel point » cet argument est-il acceptable ?* Elles attribuent à chaque argument une **force
numérique** et en déduisent un **ordre**. Deux arguments tous deux « rejetés » au sens de Dung peuvent
ainsi être **départagés** : celui qui subit trois attaques est plus faible que celui qui n'en subit qu'une.

Ce notebook présente deux méthodes déterministes, calculables à la main :

| Méthode | Principe | Référence |
|---------|----------|-----------|
| **h-Categoriser** | point fixe $\mathrm{Cat}(a) = \dfrac{1}{1 + \sum_{b \to a} \mathrm{Cat}(b)}$ | Besnard & Hunter 2001 ; convergence : Pu et al. 2014 |
| **Fardeau** (*burden-based*) | suite $\mathrm{Bur}^{i}(a) = 1 + \sum_{b \to a} \tfrac{1}{\mathrm{Bur}^{i-1}(b)}$, comparée lexicographiquement | Amgoud & Ben-Naim 2013 |

> Tout est en **C# .NET standard** (BCL seule, pas de dépendance) : les calculs sont reproductibles et déterministes.

## 1. Le cadre d'argumentation (rappel)

On réutilise exactement la structure `AF` $\langle \text{args}, \text{attacks}\rangle$ du notebook Dung :
un ensemble d'arguments et un ensemble d'attaques (couples *attaquant → attaqué*).

In [1]:
using System.Globalization;

// Cadre d'argumentation abstrait de Dung : <args, attacks>.
public sealed class AF
{
    public HashSet<string> Args { get; }
    public HashSet<(string, string)> Attaques { get; }

    public AF(IEnumerable<string> args, IEnumerable<(string, string)> attacks)
    {
        Args = new HashSet<string>(args);
        Attaques = new HashSet<(string, string)>(attacks);  // ensemble de couples (attaquant, attaque)
    }

    // Arguments qui attaquent directement x.
    public List<string> Attackers(string x) =>
        Attaques.Where(t => t.Item2 == x)
                .Select(t => t.Item1)
                .OrderBy(a => a, StringComparer.Ordinal)
                .ToList();

    public override string ToString()
    {
        var args = string.Join(", ", Args.OrderBy(a => a, StringComparer.Ordinal)
                                           .Select(a => $"'{a}'"));
        var att = string.Join(", ", Attaques.OrderBy(t => t.Item1, StringComparer.Ordinal)
                                           .ThenBy(t => t.Item2, StringComparer.Ordinal)
                                           .Select(t => $"{t.Item1}->{t.Item2}"));
        return $"AF(args=[{args}], attacks=[{att}])";
    }
}

// Exemple fil rouge : e n'est pas attaque, a subit 1 attaque, b en subit 3.
var af = new AF(
    args: new[] { "e", "a", "b", "y", "z1", "z2", "z3" },
    attacks: new[] { ("y", "a"), ("z1", "b"), ("z2", "b"), ("z3", "b") });

Console.WriteLine(af);
foreach (var x in new[] { "e", "a", "b" })
{
    var attaquants = af.Attackers(x);
    var rendu = attaquants.Count == 0
        ? "aucun"
        : string.Join(", ", attaquants.Select(a => $"'{a}'"));
    Console.WriteLine($"  attaquants de {x} : {rendu}");
}

// Assertions exécutables : cardinalités et ensembles d'attaquants du cadre fil rouge.
var argsAttendus = new HashSet<string>(new[] { "e", "a", "b", "y", "z1", "z2", "z3" });
if (af.Args.Count != 7 || !af.Args.SetEquals(argsAttendus))
{
    throw new InvalidOperationException("Le cadre fil rouge doit contenir exactement les sept arguments attendus.");
}

var attaquesAttendues = new HashSet<(string, string)>
{
    ("y", "a"),
    ("z1", "b"),
    ("z2", "b"),
    ("z3", "b")
};
if (af.Attaques.Count != 4 || !af.Attaques.SetEquals(attaquesAttendues))
{
    throw new InvalidOperationException("Le cadre fil rouge doit contenir exactement les quatre attaques attendues.");
}

var attaquantsE = af.Attackers("e");
var attaquantsA = af.Attackers("a");
var attaquantsB = af.Attackers("b");
if (attaquantsE.Count != 0 ||
    !attaquantsA.SequenceEqual(new[] { "y" }) ||
    !attaquantsB.SequenceEqual(new[] { "z1", "z2", "z3" }))
{
    throw new InvalidOperationException("Les ensembles d'attaquants de e, a et b ne correspondent pas au cadre fil rouge.");
}

Console.WriteLine("Assertions du cadre et des attaquants : CONFORME");

AF(args=['a', 'b', 'e', 'y', 'z1', 'z2', 'z3'], attacks=[y->a, z1->b, z2->b, z3->b])


  attaquants de e : aucun


  attaquants de a : 'y'


  attaquants de b : 'z1', 'z2', 'z3'


Assertions du cadre et des attaquants : CONFORME


### Lecture du résultat — lire le cadre avant de le classer

Le cadre instancié a sept arguments et quatre attaques, dont la topologie annonce déjà tout le carnet :

- **`y -> a`** : l'argument `a` subit **une** attaque, venant d'un argument (`y`) lui-même inattaqué.
- **`z1 -> b`, `z2 -> b`, `z3 -> b`** : `b` subit **trois** attaques directes, toutes de sources inattaquées.
- **`e` n'a aucun attaquant** — et n'attaque personne : c'est un argument isolé, qui servira de témoin de force maximale.
- **`z3`, `z1`, `z2`, `y`** n'apparaissent jamais à droite d'une flèche : ils sont inattaqués, donc de force maximale par construction.

`Attackers()` matérialise la seule information dont les deux sémantiques de ce carnet ont besoin : l'ensemble des attaquants directs de chaque nœud. Aucune ne regardera les cycles qui ne passent pas par ce dictionnaire.

L'asymétrie à retenir : `a` est attaqué **une fois**, `b` **trois fois**. Toute la question des sections suivantes est de savoir si cette différence (1 contre 3) produit un ordre — la sémantique de Dung ne le verra pas, les deux sémantiques graduées oui.

## 2. h-Categoriser : un point fixe d'acceptabilité

L'idée de Besnard & Hunter (2001) : la force d'un argument **décroît** avec la force cumulée de ses
attaquants. Formellement, on cherche un vecteur $\mathrm{Cat} : \text{args} \to (0, 1]$ tel que

$$\mathrm{Cat}(a) \;=\; \frac{1}{1 + \displaystyle\sum_{b \,\to\, a} \mathrm{Cat}(b)}.$$

- Un argument **non attaqué** a une force de $1$ (somme vide au dénominateur).
- Plus ses attaquants sont **nombreux** ou **forts**, plus sa force baisse vers $0$.

On calcule ce point fixe par **itération** à partir de $\mathrm{Cat}^{(0)} \equiv 1$. Pu et al. (2014) ont
prouvé que cette itération **converge** sur tout AF fini (même cyclique).

In [2]:
using System.Globalization;

// Calcule le classement h-Categoriser par iteration de point fixe.
// Renvoie (cat, n_iter) ou cat[a] in (0, 1] est la force de a (plus grand = plus fort).
public static (Dictionary<string, double> Cat, int Iterations) HCategoriser(AF af, double tol = 1e-12, int maxIter = 10_000)
{
    var cat = af.Args.ToDictionary(a => a, a => 1.0);
    for (int it = 1; it <= maxIter; it++)
    {
        var nxt = af.Args.ToDictionary(
            a => a,
            a => 1.0 / (1.0 + af.Attackers(a).Sum(b => cat[b])));
        double delta = af.Args.Count == 0
            ? 0.0
            : af.Args.Max(a => Math.Abs(nxt[a] - cat[a]));
        cat = nxt;
        if (delta < tol)
        {
            return (cat, it);
        }
    }
    return (cat, maxIter);
}

var (cat, nIter) = HCategoriser(af);
Console.WriteLine($"Convergence en {nIter} iterations.");
Console.WriteLine();
// Departage stable : force decroissante, puis nom de l'argument (ordre ordinal) pour les ex aequo.
foreach (var a in af.Args.OrderByDescending(a => cat[a]).ThenBy(a => a, StringComparer.Ordinal))
{
    Console.WriteLine($"  Cat({a}) = {cat[a].ToString("F4", CultureInfo.InvariantCulture)}");
}

// Verification programmatique du point fixe : Cat(a) = 1 / (1 + somme des forces des attaquants).
var pointFixe = af.Args.All(a =>
    Math.Abs(cat[a] - 1.0 / (1.0 + af.Attackers(a).Sum(b => cat[b]))) < 1e-9);
Console.WriteLine($"Point fixe verifie sur tous les arguments : {pointFixe}");
if (!pointFixe)
{
    throw new InvalidOperationException("Le point fixe h-Categoriser n'est pas verifie.");
}

if (!(cat["a"] > cat["b"]))
{
    throw new InvalidOperationException("h-Categoriser doit classer a strictement au-dessus de b sur l'AF fil rouge.");
}
Console.WriteLine("Departage a > b verifie : True");


Convergence en 2 iterations.


  Cat(e) = 1.0000


  Cat(y) = 1.0000


  Cat(z1) = 1.0000


  Cat(z2) = 1.0000


  Cat(z3) = 1.0000


  Cat(a) = 0.5000


  Cat(b) = 0.2500


Point fixe verifie sur tous les arguments : True


Departage a > b verifie : True


### Lecture du classement

Le classement induit (force décroissante) départe finement les arguments :

- les **non attaqués** (`e`, `y`, `z1`, `z2`, `z3`) sont au sommet à $1{,}0$ ;
- `a`, attaqué une seule fois par `y` (de force $1$), vaut $\tfrac{1}{1+1} = 0{,}5$ ;
- `b`, attaqué **trois** fois, vaut $\tfrac{1}{1+3} = 0{,}25$.

Donc `a` est **strictement mieux classé** que `b`.

## 3. Contraste avec la sémantique fondée de Dung

Calculons la sémantique **fondée** (grounded) sur le même AF — le plus petit point fixe de la
fonction caractéristique $F(S) = \{a \mid S \text{ défend } a\}$ (rappel du notebook compagnon).

In [3]:
using System.Globalization;

// Plus petit point fixe de la fonction caracteristique (semantique fondee de Dung).
public static HashSet<string> GroundedExtension(AF af)
{
    // Un ensemble S defend x si chaque attaquant b de x est battu par un membre de S.
    bool Defends(HashSet<string> s, string x) =>
        af.Attackers(x).All(b => af.Attaques.Any(t => s.Contains(t.Item1) && t.Item2 == b));

    var s = new HashSet<string>();
    while (true)
    {
        var nxt = new HashSet<string>(af.Args.Where(a => Defends(s, a)));
        if (nxt.SetEquals(s))
        {
            return s;
        }
        s = nxt;
    }
}

// Statut Dung : IN si dans l'extension, OUT si attaque par l'extension, sinon UNDEC.
public static string DungStatus(AF af, HashSet<string> g, string x)
{
    if (g.Contains(x))
    {
        return "IN";
    }
    if (af.Attaques.Any(t => g.Contains(t.Item1) && t.Item2 == x))
    {
        return "OUT";
    }
    return "UNDEC";
}

var g = GroundedExtension(af);
Console.WriteLine("Extension fondee : [" + string.Join(", ", g.OrderBy(a => a, StringComparer.Ordinal).Select(a => $"'{a}'")) + "]");
Console.WriteLine();
foreach (var x in new[] { "a", "b" })
{
    Console.WriteLine($"  {x} : statut Dung = {DungStatus(af, g, x)}  |  force h-Cat = {cat[x].ToString("F4", CultureInfo.InvariantCulture)}");
}

// Assertion d'égalité d'ensembles : l'extension doit être exactement celle du fil rouge.
var extensionAttendue = new HashSet<string>(new[] { "e", "y", "z1", "z2", "z3" });
if (!g.SetEquals(extensionAttendue))
{
    throw new InvalidOperationException("L'extension fondée du fil rouge ne correspond pas exactement à l'ensemble attendu.");
}
Console.WriteLine("Assertion de l'extension fondee : CONFORME");

Extension fondee : ['e', 'y', 'z1', 'z2', 'z3']


  a : statut Dung = OUT  |  force h-Cat = 0.5000


  b : statut Dung = OUT  |  force h-Cat = 0.2500


Assertion de l'extension fondee : CONFORME


### Le verdict

`a` et `b` ont **le même statut** dans la sémantique fondée : tous deux **OUT** (rejetés, car attaqués
par des arguments acceptés). Dung ne les distingue pas.

La sémantique de classement, elle, **les départage** : `a` ($0{,}5$) $\succ$ `b` ($0{,}25$). C'est
précisément la valeur ajoutée des sémantiques graduées — une granularité que le tout-ou-rien ne capture pas.

In [4]:
// Verification programmatique du contraste.
var memeStatutOut = DungStatus(af, g, "a") == DungStatus(af, g, "b")
                  && DungStatus(af, g, "a") == "OUT";
var hcatLesDepartage = cat["a"] > cat["b"];
Console.WriteLine($"Meme statut Dung (a, b tous deux OUT) ? {memeStatutOut}");
Console.WriteLine($"h-Categoriser les departe (Cat(a) > Cat(b)) ? {hcatLesDepartage}");

if (!memeStatutOut)
{
    throw new InvalidOperationException("Les arguments a et b doivent avoir le meme statut Dung OUT.");
}
if (!hcatLesDepartage)
{
    throw new InvalidOperationException("h-Categoriser doit strictement classer a au-dessus de b.");
}
Console.WriteLine("Assertions du contraste : CONFORME");


Meme statut Dung (a, b tous deux OUT) ? True


h-Categoriser les departe (Cat(a) > Cat(b)) ? True


Assertions du contraste : CONFORME


### Lecture du résultat — deux « True » qui disent la limite de Dung

Les deux vérifications affirment chacune une chose différente, et c'est leur conjonction qui compte.

- **Premier `True`** : `a` et `b` ont le **même** statut Dung — tous deux `OUT` (attaqués et non défendus, donc rejetés). La sémantique fondée est **binaire** : elle répond « accepté / rejeté / indécis », jamais « plus ou moins fort ». Deux arguments rejetés y sont indiscernables.
- **Second `True`** : h-Categoriser **dépare** ces deux mêmes arguments — `Cat(a) = 0,5 > Cat(b) = 0,25`.

D'où viennent ces deux nombres ? Ils sont déjà lisibles dans **la sortie de `HCategoriser`, plus haut** — pas dans les deux `True` que cette cellule vient d'afficher, qui ne portent qu'un statut et un ordre : `a` n'a qu'un attaquant, de valeur 1, donc Cat(a) = 1/(1+1) = 0,5 ; `b` a trois attaquants de valeur 1, donc Cat(b) = 1/(1+3) = 0,25. **Chaque attaquant supplémentaire divise encore la force** — le mécanisme est additif au dénominateur, pas binaire. La convergence en 2 itérations s'explique de même : les cinq arguments inattaqués valent 1 dès l'initialisation et ne bougent plus ; seule la propagation vers `a` puis `b` a besoin d'un tour.

Le contraste est donc exactement celui que le carnet veut montrer : Dung dit **qui survit**, h-Categoriser dit **combien** — et le « combien » distingue précisément ce que le statut confond.

## 4. Sémantique du fardeau (*burden-based*)

Amgoud & Ben-Naim (2013) proposent une autre construction. On définit une **suite** de nombres de
fardeau, à partir de $\mathrm{Bur}^{0}(a) = 1$ :

$$\mathrm{Bur}^{i}(a) \;=\; 1 + \sum_{b \,\to\, a} \frac{1}{\mathrm{Bur}^{i-1}(b)}.$$

Un fardeau **plus faible** signifie un argument **plus fort**. Le classement compare les
**vecteurs** $\big(\mathrm{Bur}^{1}(a), \mathrm{Bur}^{2}(a), \dots\big)$ de façon **lexicographique**
(le premier pas qui les départage tranche).

In [5]:
using System.Globalization;

// Renvoie {a: (Bur^1, ..., Bur^steps)} (Bur^0 = 1 pour tous).
public static Dictionary<string, double[]> BurdenNumbers(AF af, int steps = 8)
{
    var bur = af.Args.ToDictionary(a => a, a => 1.0);
    var seq = af.Args.ToDictionary(a => a, a => new List<double>());
    for (int i = 0; i < steps; i++)
    {
        var nxt = af.Args.ToDictionary(
            a => a,
            a => 1.0 + af.Attackers(a).Sum(b => 1.0 / bur[b]));
        foreach (var a in af.Args)
        {
            seq[a].Add(nxt[a]);
        }
        bur = nxt;
    }
    return seq.ToDictionary(kv => kv.Key, kv => kv.Value.ToArray());
}

// Comparaison lexicographique de deux vecteurs de fardeau.
public static int CompareBurden(double[] x, double[] y)
{
    int n = Math.Min(x.Length, y.Length);
    for (int i = 0; i < n; i++)
    {
        int c = x[i].CompareTo(y[i]);
        if (c != 0)
        {
            return c;
        }
    }
    return x.Length.CompareTo(y.Length);
}

// Classement par fardeau : fardeau lexicographiquement plus petit = plus fort.
public static (List<string> Order, Dictionary<string, double[]> Bn) BurdenRank(AF af, int steps = 8)
{
    var bn = BurdenNumbers(af, steps);
    var order = af.Args.OrderBy(a => a, StringComparer.Ordinal).ToList();
    order.Sort((x, y) =>
    {
        int c = CompareBurden(bn[x], bn[y]);
        return c != 0 ? c : string.CompareOrdinal(x, y);
    });
    return (order, bn);
}

var (order, bn) = BurdenRank(af);
Console.WriteLine("Classement par fardeau (du plus fort au plus faible) :");
int rang = 1;
foreach (var a in order)
{
    var v = bn[a];
    Console.WriteLine($"  {rang}. {a,-3} -> Bur^1..3 = ({v[0].ToString("F3", CultureInfo.InvariantCulture)}, " +
                                      $"{v[1].ToString("F3", CultureInfo.InvariantCulture)}, " +
                                      $"{v[2].ToString("F3", CultureInfo.InvariantCulture)})");
    rang++;
}

// Verification recalculee de la recurrence Bur^i(a) = 1 + somme 1/Bur^(i-1)(b), sur tous les degres.
double VerifierRecurrenceFardeau(AF cadre, Dictionary<string, double[]> nombres)
{
    double ecartGlobal = 0.0;
    foreach (var a in cadre.Args)
    {
        double maxEcart = 0.0;
        for (int i = 0; i < nombres[a].Length; i++)
        {
            double attendu = 1.0 + cadre.Attackers(a).Sum(b =>
                i == 0 ? 1.0 : 1.0 / nombres[b][i - 1]);
            maxEcart = Math.Max(maxEcart, Math.Abs(attendu - nombres[a][i]));
        }
        bool conforme = maxEcart < 1e-9;
        Console.WriteLine($"  {a,-3} : {(conforme ? "CONFORME" : "ECART")} (ecart max = {maxEcart.ToString("F6", CultureInfo.InvariantCulture)})");
        ecartGlobal = Math.Max(ecartGlobal, maxEcart);
    }
    return ecartGlobal;
}

Console.WriteLine();
Console.WriteLine("Controle de la recurrence du fardeau sur l'AF fil rouge :");
double ecartFilRouge = VerifierRecurrenceFardeau(af, bn);
if (ecartFilRouge >= 1e-9)
{
    throw new InvalidOperationException("La recurrence du fardeau n'est pas verifiee sur l'AF fil rouge.");
}

// Cas non trivial : sur le cycle, le fardeau d'un attaquant differe de 1.
var afCycleFardeau = new AF(new[] { "a", "b" }, new[] { ("a", "b"), ("b", "a") });
var bnCycle = BurdenNumbers(afCycleFardeau, 8);
Console.WriteLine();
Console.WriteLine("Controle de la recurrence du fardeau sur le cycle a <-> b :");
foreach (var a in afCycleFardeau.Args.OrderBy(x => x, StringComparer.Ordinal))
{
    Console.WriteLine($"  Bur^1..3({a}) = ({bnCycle[a][0].ToString("F3", CultureInfo.InvariantCulture)}, " +
                      $"{bnCycle[a][1].ToString("F3", CultureInfo.InvariantCulture)}, " +
                      $"{bnCycle[a][2].ToString("F3", CultureInfo.InvariantCulture)})");
}
double ecartCycle = VerifierRecurrenceFardeau(afCycleFardeau, bnCycle);
if (ecartCycle >= 1e-9)
{
    throw new InvalidOperationException("La recurrence du fardeau n'est pas verifiee sur le cycle.");
}
Console.WriteLine("Assertions de la recurrence du fardeau : CONFORME");


Classement par fardeau (du plus fort au plus faible) :


  1. e   -> Bur^1..3 = (1.000, 1.000, 1.000)


  2. y   -> Bur^1..3 = (1.000, 1.000, 1.000)


  3. z1  -> Bur^1..3 = (1.000, 1.000, 1.000)


  4. z2  -> Bur^1..3 = (1.000, 1.000, 1.000)


  5. z3  -> Bur^1..3 = (1.000, 1.000, 1.000)


... (12 outputs left out)

  Bur^1..3(b) = (2.000, 1.500, 1.667)


  a   : CONFORME (ecart max = 0.000000)


  b   : CONFORME (ecart max = 0.000000)


Assertions de la recurrence du fardeau : CONFORME


### Accord des deux méthodes

Sur cet AF, fardeau et h-Categoriser s'accordent sur l'essentiel : `a` reste mieux classé que `b`,
et les non-attaqués dominent. Les deux familles satisfont des **principes** communs (Amgoud & Ben-Naim) :

- **Abstraction** : le classement ne dépend que de la structure du graphe (renommer les arguments ne change rien).
- **Indépendance** : des composantes non connectées du graphe n'interfèrent pas.
- **Contre-transitivité** : *plus d'attaquants (ou des attaquants plus forts) ⇒ plus faible.*

In [6]:
// Les deux methodes s'accordent sur a > b.
var hcatOrder = af.Args.OrderByDescending(a => cat[a])
                        .ThenBy(a => a, StringComparer.Ordinal)
                        .ToList();
var hcatPlaceA = hcatOrder.IndexOf("a");
var hcatPlaceB = hcatOrder.IndexOf("b");
var burdenPlaceA = order.IndexOf("a");
var burdenPlaceB = order.IndexOf("b");
var accordHCat = hcatPlaceA < hcatPlaceB;
var accordFardeau = burdenPlaceA < burdenPlaceB;
Console.WriteLine($"h-Cat : a avant b ? {accordHCat}");
Console.WriteLine($"Fardeau : a avant b ? {accordFardeau}");

// Valeurs calculees rappelees pour que l'accord soit verifiable sans lire la sortie precedente.
Console.WriteLine($"  Cat(a) = {cat["a"].ToString("F4", CultureInfo.InvariantCulture)}, Cat(b) = {cat["b"].ToString("F4", CultureInfo.InvariantCulture)}");
Console.WriteLine($"  Bur^1(a) = {bn["a"][0].ToString("F3", CultureInfo.InvariantCulture)}, Bur^1(b) = {bn["b"][0].ToString("F3", CultureInfo.InvariantCulture)}");

if (!accordHCat)
{
    throw new InvalidOperationException("h-Categoriser doit classer a avant b.");
}
if (!accordFardeau)
{
    throw new InvalidOperationException("Le fardeau doit classer a avant b.");
}
Console.WriteLine("Assertions d'accord des deux classements : CONFORME");


h-Cat : a avant b ? True


Fardeau : a avant b ? True


  Cat(a) = 0.5000, Cat(b) = 0.2500


  Bur^1(a) = 2.000, Bur^1(b) = 4.000


Assertions d'accord des deux classements : CONFORME


### Lecture du résultat — deux échelles, un même ordre

Le classement par fardeau affiche des nombres d'une autre nature que h-Categoriser, et **la table de fardeau — produite par `BurdenNumbers` plus haut, et non par les deux `True` que cette lecture suit — permet de vérifier qu'ils mesurent la même chose autrement.

- **Les cinq arguments inattaqués** (`y`, `z3`, `z1`, `e`, `z2`) ont un fardeau constant `(1,000, 1,000, 1,000)` : sans attaquant, la somme `1 + Σ 1/Bur` vaut exactement 1 à chaque étape.
- **`a` : `(2,000, 2,000, 2,000)`** — un attaquant de fardeau 1, donc 1 + 1/1 = 2.
- **`b` : `(4,000, 4,000, 4,000)`** — trois attaquants de fardeau 1, donc 1 + 1/1 + 1/1 + 1/1 = 4.

Le fardeau est **additif et croissant avec le nombre d'attaquants**, là où h-Categoriser est **divisant et décroissant** : `Cat = 1/(1 + Σ)` contre `Bur = 1 + Σ`. Deux échelles réciproques, deux conventions de lecture (« plus grand = plus fort » dans les deux cas, mais par des formules inverses).

L'accord mesuré — `a` avant `b` pour les deux méthodes — n'est pas une coïncidence sur ce cadre : les deux formules sont **monotones** l'une de l'autre tant que les attaquants eux-mêmes restent inattaqués. Les exercices du carnet explorent l'autre cas : un cadre où l'ordre des fardeaux lexicographiques et celui des forces h-Categoriser pourraient diverger. C'est le vrai contenu du mot « sémantique » ici — une famille de fonctions, pas une réponse unique.

## 5. Le cas cyclique : convergence garantie

Sur un cycle $a \leftrightarrow b$, le tout-ou-rien de Dung laisse les deux arguments **indécidés**
(UNDEC). h-Categoriser, lui, converge vers une valeur **finie** identique pour les deux, solution de
$x = \tfrac{1}{1+x}$, soit le nombre d'or inverse $\tfrac{\sqrt{5}-1}{2} \approx 0{,}618$.

In [7]:
using System.Globalization;

// Construire un cadre cyclique symetrique : a et b s'attaquent mutuellement.
var afCycle = new AF(new[] { "a", "b" }, new[] { ("a", "b"), ("b", "a") });

// Appliquer h-Categoriser pour calculer le point fixe et le nombre d'iterations.
var (catCyc, itCyc) = HCategoriser(afCycle);

// Comparer la valeur calculee a la solution analytique de x = 1 / (1 + x).
double phiInv = (Math.Sqrt(5) - 1) / 2;
Console.WriteLine($"AF cyclique a<->b : convergence en {itCyc} iterations.");
Console.WriteLine($"  Cat(a) = {catCyc["a"].ToString("F6", CultureInfo.InvariantCulture)}, " +
                  $"Cat(b) = {catCyc["b"].ToString("F6", CultureInfo.InvariantCulture)}");
Console.WriteLine($"  Valeur theorique (sqrt(5)-1)/2 = {phiInv.ToString("F6", CultureInfo.InvariantCulture)}");

// Calculer l'extension fondee, vide sur ce cycle, puis determiner le statut Dung.
var gCycle = GroundedExtension(afCycle);
string extRendu = gCycle.Count == 0
    ? "vide"
    : "[" + string.Join(", ", gCycle.OrderBy(a => a, StringComparer.Ordinal)) + "]";
Console.WriteLine($"  Statut Dung (fonde) des deux : {DungStatus(afCycle, gCycle, "a")} " +
                  $"(extension fondee = {extRendu} )");

// Assertions executables : une divergence doit faire echouer l'execution.
const double toleranceCycle = 1e-9;
if (itCyc != 29)
{
    throw new InvalidOperationException($"Le cycle doit converger en 29 iterations, mais il en faut {itCyc}.");
}
if (Math.Abs(catCyc["a"] - phiInv) >= toleranceCycle ||
    Math.Abs(catCyc["b"] - phiInv) >= toleranceCycle)
{
    throw new InvalidOperationException("Les scores calcules du cycle divergent de la solution analytique.");
}
if (Math.Abs(catCyc["a"] - catCyc["b"]) >= toleranceCycle)
{
    throw new InvalidOperationException("Les deux arguments du cycle symetrique doivent avoir le meme score.");
}
if (gCycle.Count != 0)
{
    throw new InvalidOperationException("L'extension fondee du cycle doit etre vide.");
}
if (DungStatus(afCycle, gCycle, "a") != "UNDEC" ||
    DungStatus(afCycle, gCycle, "b") != "UNDEC")
{
    throw new InvalidOperationException("Les deux arguments du cycle doivent avoir le statut Dung UNDEC.");
}
Console.WriteLine("Assertions du cycle : CONFORME");


AF cyclique a<->b : convergence en 29 iterations.


  Cat(a) = 0.618034, Cat(b) = 0.618034


  Valeur theorique (sqrt(5)-1)/2 = 0.618034


  Statut Dung (fonde) des deux : UNDEC (extension fondee = vide )


Assertions du cycle : CONFORME


### Lecture du résultat — le cycle : convergence garantie, verdict partagé

Le cas cyclique est le test de robustesse des deux sémantiques, et la sortie le tranche proprement.

**Côté Dung**, l'extension fondée est **vide** : `a` attaque `b` qui attaque `a`, aucun des deux ne peut être défendu sans présupposer l'autre. Les deux arguments sont `UNDEC` — indécis, pas rejetés. La sémantique binaire refuse de choisir.

**Côté h-Categoriser**, l'itération converge toujours, mais en **29 itérations** contre 2 sur le cadre acyclique — le cycle ralentit la contraction sans l'empêcher. Et le point fixe est remarquable : `Cat(a) = Cat(b) = 0,618034`, valeur qui coïncide exactement avec **(√5−1)/2**, l'inverse du nombre d'or.

Ce n'est pas un hasard numérique : la récurrence s'écrit `x = 1/(1+x)` quand `a` et `b` se valent (chacun a exactement un attaquant, de même force). En multipliant par `(1+x)`, on obtient `x² + x − 1 = 0`, dont la racine positive est précisément `(√5−1)/2 ≈ 0,618034`. La sortie affiche la valeur théorique à côté de la valeur calculée — l'accord à six décimales **vérifie l'implémentation** du point fixe.

**Le verdict partagé** : sur un cycle symétrique, h-Categoriser **ne crée pas** d'ordre que Dung aurait manqué — les deux arguments restent à égalité stricte (0,618034 chacun). La différence entre les deux sémantiques ne porte donc pas sur l'existence d'un ordre, mais sur sa **finesse** : tout deux-capables quand les défaites sont symétriques, départage seulement quand le nombre d'attaquants diffère (le cas `a` contre `b` de la section 3).

## 6. Exercices

> Les cellules ci-dessous sont des **squelettes à compléter**. Le notebook s'exécute de bout en bout
> même si vous ne les complétez pas (elles renvoient `null` par défaut).

### Exercice 1 — Départage strict

Complétez `IsStrictlyStronger(cat, a, b)` : renvoyer `true` si `a` est **strictement** mieux classé
que `b` selon le vecteur `cat` de h-Categoriser, `false` sinon.

In [8]:
// Exercice 1 — Départage strict
public static bool? IsStrictlyStronger(Dictionary<string, double> cat, string a, string b)
{
    // TODO etudiant : comparer cat[a] et cat[b]
    // Indice : "strictement mieux classe" = force strictement superieure.
    return null; // remplacez par votre comparaison
}

// Test attendu (une fois complete) : True, puis False.
var resultaSurB = IsStrictlyStronger(cat, "a", "b");
var resultbSurA = IsStrictlyStronger(cat, "b", "a");

// Les assertions ne s'exécutent que lorsque le stub a été complété.
if (resultaSurB.HasValue && resultbSurA.HasValue)
{
    if (!resultaSurB.Value)
    {
        throw new InvalidOperationException("Après complétion, a doit être strictement mieux classé que b.");
    }
    if (resultbSurA.Value)
    {
        throw new InvalidOperationException("Après complétion, b ne doit pas être strictement mieux classé que a.");
    }
    Console.WriteLine("Assertions du départage strict : CONFORME");
}
else
{
    Console.WriteLine($"a strictement > b ? {resultaSurB?.ToString() ?? "None"}");
    Console.WriteLine($"b strictement > a ? {resultbSurA?.ToString() ?? "None"}");
}

a strictement > b ? None


b strictement > a ? None


### Exercice 2 — Construire un cas de départage

Définissez un AF `afEx2` à **4 arguments** dans lequel h-Categoriser **départage** deux arguments que
la sémantique fondée déclare tous deux rejetés (OUT).

*Indice : reprenez le motif de la section 3 — un argument attaqué une fois, un autre attaqué deux fois,
par des arguments non attaqués.*

In [9]:
#nullable enable

// Exercice 2 — Construire un cas de departage
using System.Globalization;

// TODO etudiant : construire afEx2 = new AF(new[] { ... }, new[] { ... })
AF? afEx2 = null; // remplacez par votre cadre

if (afEx2 is not null)
{
    var (c2, _) = HCategoriser(afEx2);
    var forces = string.Join(", ", c2.OrderBy(kv => kv.Key, StringComparer.Ordinal)
                               .Select(kv => $"{kv.Key}: {Math.Round(kv.Value, 3).ToString("F3", CultureInfo.InvariantCulture)}"));
    Console.WriteLine("Forces h-Cat : " + forces);

    // Assertions exécutables : le cadre doit contenir quatre arguments et
    // présenter deux arguments OUT strictement départagés par h-Categoriser.
    if (afEx2.Args.Count != 4)
    {
        throw new InvalidOperationException("Le cadre de l'exercice 2 doit contenir exactement quatre arguments.");
    }

    var g2 = GroundedExtension(afEx2);
    var pairesDepartagees = afEx2.Args
        .SelectMany(x => afEx2.Args.Where(y => !string.Equals(x, y, StringComparison.Ordinal))
                                  .Select(y => (x, y)))
        .Where(pair => DungStatus(afEx2, g2, pair.x) == "OUT" &&
                       DungStatus(afEx2, g2, pair.y) == "OUT" &&
                       c2[pair.x] > c2[pair.y])
        .ToList();

    if (pairesDepartagees.Count == 0)
    {
        throw new InvalidOperationException("Le cadre doit contenir deux arguments OUT strictement départagés par h-Categoriser.");
    }
    Console.WriteLine("Assertions du cas de départage : CONFORME");
}
else
{
    Console.WriteLine("Exercice 2 a completer");
}

Exercice 2 a completer


### Exercice 3 — Détecter la convergence du fardeau

Complétez `BurdenConverged(af, steps, tol)` : renvoyer `true` si, sur l'AF donné, le **dernier** pas
de la suite de fardeau ne diffère du précédent que de moins de `tol` pour **tous** les arguments.

*Indice : `BurdenNumbers(af, steps)` renvoie les vecteurs ; comparez les deux dernières composantes.*

In [10]:
// Exercice 3 — Détecter la convergence du fardeau
public static bool? BurdenConverged(AF af, int steps = 8, double tol = 1e-6)
{
    // TODO etudiant : comparer les deux derniers pas de chaque vecteur de fardeau
    // bn = BurdenNumbers(af, steps) ; bn[a] est un tableau de longueur steps.
    return null; // remplacez par votre test de convergence
}

var convergenceFardeau = BurdenConverged(af);
if (convergenceFardeau.HasValue)
{
    if (!convergenceFardeau.Value)
    {
        throw new InvalidOperationException("Après complétion, le fardeau de l'AF fil rouge doit être détecté comme convergent.");
    }
    Console.WriteLine("Assertion de convergence du fardeau : CONFORME");
}
else
{
    Console.WriteLine($"Fardeau convergent sur l'AF fil rouge ? {convergenceFardeau?.ToString() ?? "None"}");
}

Fardeau convergent sur l'AF fil rouge ? None


## À retenir

- Les sémantiques **de classement** raffinent le tout-ou-rien de Dung en attribuant une **force
  numérique** à chaque argument, ce qui **départage** des arguments de même statut — mesuré sur le fil
  rouge : `a` et `b` tous deux OUT pour Dung, mais `Cat(a) = 0.5000` contre `Cat(b) = 0.2500`.
- **h-Categoriser** ($\mathrm{Cat}(a) = 1/(1+\sum_{b\to a}\mathrm{Cat}(b))$) est un point fixe qui
  **converge** sur tout AF fini, y compris cyclique.
- La sémantique du **fardeau** classe par comparaison **lexicographique** d'une suite de nombres.
- Les deux respectent des **principes** structurels (abstraction, indépendance, contre-transitivité).

## Vérifiez votre compréhension

**Question 1 — Pourquoi la sortie de la section 3 imprime-t-elle deux `True` plutôt qu'un seul ?**

La cellule imprime `Meme statut Dung (a, b tous deux OUT) ? True` puis `h-Categoriser les departe (Cat(a) > Cat(b)) ? True`.

<details><summary>Réponse</summary>

Parce que ce sont **deux propositions différentes** : la première dit que la sémantique fondée de Dung ne distingue pas `a` (attaqué par 1 argument) de `b` (attaqué par 3) — tous deux OUT ; la seconde dit que h-Categoriser, lui, les départage (`Cat(a) = 0.5` contre `Cat(b) = 0.25`). Ensemble, les deux `True` **mesurent la limite de Dung** : c'est le contraste exact entre statut tout-ou-rien et force graduée — et pourquoi les sémantiques de classement existent.

</details>

**Question 2 — Le classement par fardeau rend `a -> (2.000, 2.000, 2.000)` et `b -> (4.000, 4.000, 4.000)` : que signifie le triplet, et pourquoi suffit-il à départager ?**

La section 4 imprime le classement complet `Bur^1..3` pour les sept arguments du fil rouge, et la cellule d'accord conclut `h-Cat : a avant b ? True` / `Fardeau : a avant b ? True`.

<details><summary>Réponse</summary>

Chaque composante `Bur^k` est la valeur de fardeau de l'argument à l'itération `k`, obtenue par la récurrence `Bur^k(a) = 1 + Σ 1/Bur^(k-1)(b)`. Ce n'est pas un compte de chaînes d'attaques de longueur `k` : sur le fil rouge, `Bur^2(a) = 2` alors qu'aucune chaîne de longueur 2 n'aboutit à `a`. La comparaison est **lexicographique** : le premier degré où les suites diffèrent suffit à départager. Ici, `a` a un fardeau constant de `2.000`, tandis que `b` a un fardeau constant de `4.000` ; l'accord imprimé avec h-Cat (`a` avant `b` des deux côtés) montre deux échelles numériques différentes qui produisent le même ordre sur cet exemple.

</details>

**Question 3 — Sur le cycle a ↔ b, pourquoi citer la valeur théorique $(\sqrt{5}-1)/2$ à côté de la valeur mesurée ?**

La section 5 imprime `AF cyclique a<->b : convergence en 29 iterations`, `Cat(a) = Cat(b) = 0.618034`, `Valeur theorique (sqrt(5)-1)/2 = 0.618034`, et `Statut Dung (fonde) des deux : UNDEC`.

<details><summary>Réponse</summary>

Parce que le cycle donne une équation de point fixe **résoluble analytiquement** : Cat = 1/(1+Cat) a pour solution l'inverse du nombre d'or. La valeur mesurée à 10⁻⁶ près après 29 itérations prouve la convergence vers ce point fixe — la théorie prédit, l'itération vérifie. Et le contraste est complet : là où Dung rend les deux arguments UNDEC (extension fondée vide), le classement rend une force **finie et égale** pour chacun. Les trois exercices prolongent ce travail : départage strict, cas de départage à construire, détection de convergence du fardeau.

</details>

## Attestation de parité Python / C#

Ce jumeau rejoue le fil du carnet Python avec la BCL .NET (`HashSet`, `Dictionary`, LINQ). Les
valeurs ci-dessous sont **celles produites par l'exécution de ce carnet** et coïncident terme à terme
avec la source Python :

| Mesure | Python | C# (ce carnet) |
|--------|--------|----------------|
| Cadre fil rouge | 7 arguments, 4 attaques | 7 arguments, 4 attaques |
| Extension fondée | `['e', 'y', 'z1', 'z2', 'z3']` | `['e', 'y', 'z1', 'z2', 'z3']` |
| Statuts de `a` et `b` | `OUT` / `OUT` | `OUT` / `OUT` |
| Forces h-Categoriser | `Cat(a) = 0.5000`, `Cat(b) = 0.2500`, non attaqués à `1.0000` | identiques |
| Itérations de convergence (cadre acyclique) | 2 | 2 |
| Vecteurs de fardeau | `a -> (2.000, 2.000, 2.000)`, `b -> (4.000, 4.000, 4.000)`, non attaqués à `(1.000, 1.000, 1.000)` | identiques |
| Cycle `a ↔ b` | `Cat(a) = Cat(b) = 0.618034`, valeur théorique `(√5−1)/2 = 0.618034`, extension fondée vide, `UNDEC` | identiques |
| Itérations de convergence (cycle) | 29 | 29 |
| Accords de classement | `h-Cat : a avant b ? True`, `Fardeau : a avant b ? True` | `h-Cat : a avant b ? True`, `Fardeau : a avant b ? True` |

Les contrôles exécutables des cellules de calcul vérifient le point fixe, le contraste Dung/h-Categoriser,
la récurrence du fardeau sur le fil rouge et sur le cycle, l'accord des deux classements et toutes les
propriétés annoncées du cycle. Les trois stubs d'exercice restent neutres (`null`) : le carnet s'exécute de bout en bout sans solution révélée, et les valeurs du tableau proviennent uniquement des cellules de calcul.

## Pour aller plus loin

- [Jumeau Python](Argumentation-03c-Ranking-Semantics-Python.ipynb) — le carnet source, dont ce portage rejoue les résultats.
- [Index de la sous-série ArgumentAnalysis](README.md) — la place des sémantiques de classement dans le parcours.
- [Index de la série SymbolicAI](../README.md) — le fil complet, du raisonnement formel aux systèmes multi-agents.

La bibliothèque **Tweety** (`org.tweetyproject.arg.rankings`) implémente ces sémantiques et bien d'autres (Categoriser, Burden, Discussion-based, Tuples\*, Matt & Toni). Voir aussi le notebook compagnon [`Argumentation-03-Dung-AF-Semantics`](Argumentation-03-Dung-AF-Semantics-Python.ipynb) pour les sémantiques d'extension, et [`Argumentation-03b-Value-Based-AF`](Argumentation-03b-Value-Based-AF-Python.ipynb) pour une autre voie de départage (préférences d'audience plutôt que force numérique).

**Références.**
- P. Besnard, A. Hunter (2001). *A logic-based theory of deductive arguments*. Artificial Intelligence.
- L. Amgoud, J. Ben-Naim (2013). *Ranking-based semantics for argumentation frameworks*. SUM.
- F. Pu, J. Luo, Y. Zhang, G. Luo (2014). *Argument ranking with categoriser function*. KSEM (convergence).

***

*Notebook pédagogique — C# (.NET, BCL seule), déterministe, sans LLM ni solveur externe.*